# 植物幼苗分类 V2：带约束的残差分类头

**使用方法**：把本 `.ipynb` 导入 Kaggle → 添加 `2026task1` 原始比赛数据 →
Settings 中开启 **GPU** 和 **Internet** → Run All。最终提交文件是
**`/kaggle/working/submission.csv`**。Notebook 自带全部代码，不需要上传参考代码包。

V2 采用独立运行的完整流程，在 Plant V3 方法基础上加入受限残差分类头与分离的训练目标。

| 阶段 | 方法 | 作用 |
|---|---|---|
| 通用五分类 | DINOv2-Large、518 像素主体裁剪、翻转特征平均、五折 RBF SVM | 识别五种植物 |
| 草类专家 | 两个骨干均使用 Linear + 64 维非线性残差头 | 增加小规模的非线性修正能力 |
| 训练目标 | 原分支交叉熵 + 残差分支交叉熵与 KL 约束，梯度隔离 | 让新增分支学习时不反向改变原分支 |
| 融合 | 两专家各 0.5，Loose Silky-bent 阈值 0.25 | 沿用参考方案的统一规则 |
| 几何匹配 | RootSIFT + RANSAC，检索带标签训练图片 | 利用重复拍摄场景辅助纠正草类 |

V2 正式公榜为 **0.9841**。0.25 阈值与固定训练轮数沿用既定配方。

默认按阶段顺序使用 **一张 GPU**，T4 x2 也能运行但只使用第 0 张，便于排错、恢复训练和控制显存。
训练耗时以实际进度条为准。除最终 `submission.csv` 外，还导出 `submission_v2.csv`、
`submission_baseline.csv` 和差异表，一次训练即可观察新增分支对预测的影响。




## 1. 依赖准备

保留 Kaggle 预装的匹配版 PyTorch / torchvision，避免替换 CUDA 环境。
Transformers 固定为 4.57.1。只在版本不符或缺包时安装；首次运行需 Internet。
若曾在同一内核导入其他版本的 Transformers，安装后重启内核再 Run All。


In [ ]:
import importlib.metadata as metadata
import importlib.util
import subprocess
import sys

def installed_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

packages = []
if installed_version("transformers") != "4.57.1":
    packages.append("transformers==4.57.1")
for module, package in [
    ("numpy", "numpy"), ("pandas", "pandas"), ("sklearn", "scikit-learn"),
    ("cv2", "opencv-python-headless"), ("PIL", "Pillow"), ("joblib", "joblib"),
    ("tqdm", "tqdm"), ("matplotlib", "matplotlib"),
]:
    if importlib.util.find_spec(module) is None:
        packages.append(package)
if packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("Dependency setup complete.")


## 2. 参数集中配置

通常不用改。自动定位失败时，把 `DATA_ROOT` 填为含 `train/` 和 `test/` 的目录；
`SAMPLE_SUBMISSION` 可指定提交模板。支持 png/jpg/jpeg/bmp/webp，图片数量从数据读取。
原作业预计有 500 张训练图、378 张测试图，数量不同会提示但不写死数组大小。

`RESUME_FROM` 用于跨 Kaggle 会话续训：先保存旧 Notebook 输出，再把它作为输入添加，
填入旧输出的 `plant_rewrite_v2` 文件夹或 `run_...` 文件夹。续训使用本版检查点。
相同数据和配置下，特征、SVM 和每轮专家训练会自动恢复。修改配置会生成新的运行目录。
当前 GPU 显存不足时，可减小 `FEATURE_BATCH` / `TRAIN_BATCH`，后者配合增大 `ACCUMULATION`。
`USE_RETRIEVAL=False` 可以跳过 CPU 图像匹配；`TRAIN_EXPERTS=False` 可以先跑特征基线。
`ADAPTER_WIDTH=64` 为新增分支宽度，`LOGIT_CORRECTION_LIMIT=0.10` 为每个 logit 的修正上限。
默认不增大上限。若设为 0，则新分支不改变预测，可作为消融实验。


In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
import copy
import gc
import hashlib
import json
import math
import os
import random
import shutil
import time

import cv2
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from tqdm.auto import tqdm
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T
from transformers import AutoConfig, AutoModel
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import normalize
from sklearn.svm import SVC
from IPython.display import display, FileLink

@dataclass
class Settings:
    DATA_ROOT: str = ""
    SAMPLE_SUBMISSION: str = ""
    RESUME_FROM: str = ""
    OUTPUT_ROOT: str = "/kaggle/working/plant_rewrite_v2"
    CACHE_ROOT: str = "/kaggle/temp/plant_rewrite_v2"
    CLASSES: tuple = ("Black-grass", "Common wheat", "Loose Silky-bent", "Scentless Mayweed", "Sugar beet")
    FOLD_SEED: int = 20260920
    TRAIN_SEED: int = 20260921
    FOLDS: int = 5
    IMAGE_SIZE: int = 518
    FEATURE_BATCH: int = 4
    TRAIN_BATCH: int = 8
    ACCUMULATION: int = 2
    WORKERS: int = 2
    SVM_C: float = 10.0
    TRAIN_EXPERTS: bool = True
    DINO_EPOCHS: int = 15
    CONVNEXT_EPOCHS: int = 10
    # 保持原参考训练轨迹：原计划 30 轮，前 3 轮冻结，后 27 轮使用余弦衰减。
    COSINE_T_MAX: int = 27
    ADAPTER_WIDTH: int = 64
    LOGIT_CORRECTION_LIMIT: float = 0.10
    ADAPTER_LR: float = 1e-3
    ADAPTER_LOSS_WEIGHT: float = 0.5
    DISTILL_WEIGHT: float = 1.0
    DISTILL_TEMPERATURE: float = 2.0
    GRASS_THRESHOLD: float = 0.25
    USE_RETRIEVAL: bool = True
    MATCH_MIN_SCORE: float = 10.0
    MATCH_MARGIN: float = 1.5
    # 可换成本地 Hugging Face save_pretrained 目录；默认首次联网下载。
    DINO_LARGE_SOURCE: str = "facebook/dinov2-large"
    DINO_BASE_SOURCE: str = "facebook/dinov2-base"
    # 留空使用官方 ImageNet 预训练权重；离线可填官方 ConvNeXt-Tiny .pth。
    CONVNEXT_WEIGHTS: str = ""

CFG = Settings()
PIPELINE_VERSION = "plant-rewrite-2.0-residual-head"
assert CFG.IMAGE_SIZE % 14 == 0, "DINOv2 输入边长应为 14 的倍数"
assert CFG.FOLDS >= 2 and CFG.FEATURE_BATCH > 0 and CFG.TRAIN_BATCH > 0
assert CFG.ACCUMULATION >= 1 and 0 < CFG.GRASS_THRESHOLD < 1
assert 1 <= CFG.DINO_EPOCHS <= 30 and 1 <= CFG.CONVNEXT_EPOCHS <= 30
assert CFG.ADAPTER_WIDTH > 0 and 0 <= CFG.LOGIT_CORRECTION_LIMIT <= 0.5
assert CFG.ADAPTER_LR > 0 and CFG.ADAPTER_LOSS_WEIGHT > 0
assert CFG.DISTILL_WEIGHT >= 0 and CFG.DISTILL_TEMPERATURE > 0
assert torch.cuda.is_available(), "请在 Kaggle Settings → Accelerator 中开启 GPU 后重跑。"
DEVICE = torch.device("cuda:0")
torch.set_num_threads(2)
cv2.setNumThreads(2)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def atomic_json(path, data):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
    tmp.replace(path)

def atomic_torch_save(path, data):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(data, tmp)
    tmp.replace(path)

def hash_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

seed_everything(CFG.TRAIN_SEED)
ENVIRONMENT = {name: installed_version(name) for name in [
    "torch", "torchvision", "transformers", "numpy", "pandas", "scikit-learn",
    "opencv-python-headless", "opencv-python", "Pillow", "joblib",
]}
print("Using", torch.cuda.get_device_name(0), "| available GPUs:", torch.cuda.device_count())
print(ENVIRONMENT)


## 3. 查找图片与提交模板

只从训练图片的文件夹名读取标签。提交模板只提供 ID 及顺序，**不使用其 Category 内容**。
如果有多个相同 ID 集合的模板，优先 `sample_submission`；模板缺失或存在歧义时明确报错。
若数据只提供 zip，把其中的 `train/`、`test/` 先解压成 Kaggle Dataset 后挂载。
对每张图片计算内容哈希，防止数据变化后误用旧缓存。


In [ ]:
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

def list_images(directory):
    return sorted(p for p in Path(directory).rglob("*")
                  if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)

def locate_data(config, input_root=Path("/kaggle/input")):
    if config.DATA_ROOT:
        root = Path(config.DATA_ROOT)
        if not all((root / split).is_dir() for split in ["train", "test"]):
            raise ValueError(f"DATA_ROOT 必须同时包含 train/ 和 test/: {root}")
        if not all((root / "train" / label).is_dir() for label in config.CLASSES):
            raise ValueError("训练目录缺少配置中的类别文件夹")
        return root
    candidates = []
    for folder in input_root.rglob("train"):
        if folder.is_dir() and (folder.parent / "test").is_dir():
            if all((folder / label).is_dir() for label in config.CLASSES):
                candidates.append(folder.parent)
    if len(candidates) != 1:
        raise ValueError(f"找到 {len(candidates)} 组数据: {candidates}。请填写 CFG.DATA_ROOT。")
    return candidates[0]

def read_submission_ids(path, expected_ids):
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    if "ID" not in frame.columns or "Category" not in frame.columns:
        raise ValueError(f"模板须包含 ID、Category 列: {path}")
    ids = frame.ID.tolist()
    if len(ids) != len(set(ids)) or set(ids) != set(expected_ids):
        raise ValueError(f"模板 ID 重复或与 test 图片不一致: {path}")
    return ids

def locate_template(root, config, expected_ids, input_root=Path("/kaggle/input")):
    if config.SAMPLE_SUBMISSION:
        path = Path(config.SAMPLE_SUBMISSION)
        return path, read_submission_ids(path, expected_ids)
    valid = []
    for path in input_root.rglob("*.csv"):
        if "submission" not in path.name.lower():
            continue
        try:
            ids = read_submission_ids(path, expected_ids)
        except (ValueError, pd.errors.ParserError, UnicodeError):
            continue
        priority = (0 if root in path.parents else 1,
                    0 if "sample" in path.name.lower() else 1)
        valid.append((priority, path, ids))
    if not valid:
        raise FileNotFoundError("未找到 ID 匹配的提交模板，请设置 CFG.SAMPLE_SUBMISSION。")
    valid.sort(key=lambda item: (item[0], str(item[1])))
    peers = [item for item in valid if item[0] == valid[0][0]]
    if any(item[2] != peers[0][2] for item in peers):
        raise ValueError("多个同优先级模板的顺序不同，请指定 CFG.SAMPLE_SUBMISSION。")
    return valid[0][1], valid[0][2]

data_root = locate_data(CFG)
train_rows = []
for label, category in enumerate(CFG.CLASSES):
    for path in list_images(data_root / "train" / category):
        train_rows.append({"ID": path.name, "path": str(path), "label": label, "Category": category})
train = pd.DataFrame(train_rows)
assert len(train) and train.groupby("label").size().reindex(range(len(CFG.CLASSES)), fill_value=0).min() >= CFG.FOLDS
test_paths = list_images(data_root / "test")
test_map = {path.name: path for path in test_paths}
assert test_paths and len(test_map) == len(test_paths), "test 中存在同名文件或没有图片"
template_path, template_ids = locate_template(data_root, CFG, test_map)
test = pd.DataFrame({"ID": template_ids, "path": [str(test_map[name]) for name in template_ids]})
for frame in [train, test]:
    frame["sha256"] = [hash_file(path) for path in tqdm(frame.path, desc="Checking image content")]
train["fold"] = -1
splitter = StratifiedKFold(CFG.FOLDS, shuffle=True, random_state=CFG.FOLD_SEED)
for fold, (_, valid_indices) in enumerate(splitter.split(train, train.label)):
    train.loc[valid_indices, "fold"] = fold

# 内容、标签、行顺序、训练参数及依赖版本共同确定缓存目录；挂载路径改变不影响数据身份。
run_config = asdict(CFG)
for key in ["DATA_ROOT", "SAMPLE_SUBMISSION", "RESUME_FROM", "OUTPUT_ROOT", "CACHE_ROOT"]:
    run_config.pop(key)
identity = {
    "pipeline": PIPELINE_VERSION, "config": run_config, "environment": ENVIRONMENT,
    "train": train[["ID", "label", "fold", "sha256"]].to_dict("records"),
    "test": test[["ID", "sha256"]].to_dict("records"),
}
RUN_ID = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:16]
OUT = Path(CFG.OUTPUT_ROOT) / f"run_{RUN_ID}"
OUT.mkdir(parents=True, exist_ok=True)
if CFG.RESUME_FROM:
    previous = Path(CFG.RESUME_FROM)
    if previous.name != OUT.name:
        previous = previous / OUT.name
    if not (previous / "identity.json").exists():
        raise ValueError(f"没有找到匹配本次数据、配置及依赖版本的输出: {previous}")
    if json.loads((previous / "identity.json").read_text()) != json.loads(json.dumps(identity)):
        raise ValueError("恢复目录与当前运行不匹配")
    # 只复制本次尚未写入的文件，重新 Run All 不会覆盖更新的断点。
    for source in tqdm(list(previous.rglob("*")), desc="Restoring saved output"):
        destination = OUT / source.relative_to(previous)
        if source.is_file() and not destination.exists() and not source.name.endswith(".tmp"):
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
CACHE = Path(CFG.CACHE_ROOT) / RUN_ID
CACHE.mkdir(parents=True, exist_ok=True)
atomic_json(OUT / "identity.json", identity)
atomic_json(OUT / "config.json", asdict(CFG))
atomic_json(OUT / "environment.json", ENVIRONMENT)
print(f"Data: {data_root}\nTemplate: {template_path}\nOutput: {OUT}")
print(f"Train: {len(train)}, test: {len(test)}")
if (len(train), len(test)) != (500, 378):
    print("提示：数量与参考作业 500/378 不同，按实际图片数量继续。")
display(pd.crosstab(train.Category, train.fold))


## 4. 生成三种图像视角

`raw` 保留完整图像；`crop` 用 HSV 与超绿指数寻找植物区域后留边裁剪；
`masked` 在裁剪的基础上柔和降低背景影响。全部等比例缩放并填充到 518×518。
没有检测到足够绿色像素时退回原图，避免把整张图片抹掉。
此步骤没有学习参数，训练与测试使用完全一致的处理。


In [ ]:
def make_views(path, size):
    with Image.open(path) as source:
        raw = ImageOps.exif_transpose(source).convert("RGB")
        raw.thumbnail((1600, 1600), Image.Resampling.LANCZOS)
    rgb = np.asarray(raw)
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    channels = rgb.astype(np.int16)
    excess_green = 2 * channels[:, :, 1] - channels[:, :, 0] - channels[:, :, 2]
    green = ((hsv[:, :, 0] >= 25) & (hsv[:, :, 0] <= 100) &
             (hsv[:, :, 1] >= 30) & (hsv[:, :, 2] >= 20) & (excess_green > 12)).astype(np.uint8)
    count, component_map, stats, _ = cv2.connectedComponentsWithStats(green, connectivity=8)
    kept = np.zeros_like(green)
    if count > 1:
        cutoff = max(5, int(stats[1:, cv2.CC_STAT_AREA].max() * 0.01))
        keep_labels = np.flatnonzero(stats[:, cv2.CC_STAT_AREA] >= cutoff)
        keep_labels = keep_labels[keep_labels != 0]
        kept = np.isin(component_map, keep_labels).astype(np.uint8)
    ys, xs = np.where(kept)
    box = (0, 0, raw.width, raw.height)
    if len(xs) >= max(12, int(raw.width * raw.height * 0.001)):
        margin = max(8, int(max(xs.max() - xs.min(), ys.max() - ys.min()) * 0.15))
        box = (max(0, int(xs.min()) - margin), max(0, int(ys.min()) - margin),
               min(raw.width, int(xs.max()) + margin + 1), min(raw.height, int(ys.max()) + margin + 1))
        alpha = cv2.dilate(kept, np.ones((7, 7), np.uint8)).astype(np.float32)
        alpha = cv2.GaussianBlur(alpha, (9, 9), 0)
    else:
        alpha = np.ones_like(kept, dtype=np.float32)
    alpha = (0.25 + 0.75 * alpha)[:, :, None]
    softened = Image.fromarray(np.clip(rgb * alpha + 110 * (1 - alpha), 0, 255).astype(np.uint8))
    padding = tuple(np.median(rgb.reshape(-1, 3), axis=0).astype(int).tolist())
    variants = {"raw": raw, "crop": raw.crop(box), "masked": softened.crop(box)}
    return {name: ImageOps.pad(image, (size, size), method=Image.Resampling.LANCZOS, color=padding)
            for name, image in variants.items()}

for split, frame in [("train", train), ("test", test)]:
    for index, row in tqdm(frame.iterrows(), total=len(frame), desc=f"Preparing {split}"):
        paths = {view: CACHE / f"{split}_{index:05d}_{view}.png" for view in ["raw", "crop", "masked"]}
        if not all(path.exists() for path in paths.values()):
            for view, image in make_views(row.path, CFG.IMAGE_SIZE).items():
                temporary = paths[view].with_suffix(".tmp")
                image.save(temporary, format="PNG", compress_level=1)
                temporary.replace(paths[view])
        for view, path in paths.items():
            frame.loc[index, f"{view}_path"] = str(path)
train.to_csv(OUT / "train_manifest.csv", index=False)
test.to_csv(OUT / "test_manifest.csv", index=False)

fig, axes = plt.subplots(len(CFG.CLASSES), 3, figsize=(9, 3 * len(CFG.CLASSES)))
for row_index, category in enumerate(CFG.CLASSES):
    row = train[train.Category == category].iloc[0]
    for col, view in enumerate(["raw", "crop", "masked"]):
        with Image.open(row[f"{view}_path"]) as image:
            axes[row_index, col].imshow(image)
        axes[row_index, col].set_title(f"{category}\n{view}")
        axes[row_index, col].axis("off")
plt.tight_layout()
plt.show()


## 5. 冻结 DINOv2-Large 提取特征

每张图取 CLS token 与所有 patch token 的均值，分别 L2 归一化后拼接。
原图和水平翻转图的向量取平均，再归一化；最终每张图得到 2048 维特征。
只计算最终方案需要的 `crop` 视角，不再穷举所有骨干与视角。
骨干权重、来源版本、特征及样本顺序都保存到输出目录。

模型接口参考：[Transformers DINOv2 文档](https://huggingface.co/docs/transformers/v4.57.1/en/model_doc/dinov2)。


In [ ]:
NORMALIZE = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

def seed_worker(_):
    seed = torch.initial_seed() % (2 ** 32)
    random.seed(seed)
    np.random.seed(seed)
    cv2.setNumThreads(0)

def make_loader(dataset, batch_size, shuffle=False, generator=None):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      num_workers=CFG.WORKERS, pin_memory=True,
                      worker_init_fn=seed_worker, generator=generator,
                      persistent_workers=False)

class ViewDataset(Dataset):
    def __init__(self, frame, view="crop", size=None):
        self.paths = frame[f"{view}_path"].tolist()
        steps = ([T.Resize((size, size))] if size else []) + [T.ToTensor(), NORMALIZE]
        self.transform = T.Compose(steps)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            return self.transform(image.convert("RGB"))

def load_backbone(kind, dtype=torch.float32):
    destination = OUT / f"dinov2_{kind}"
    complete = destination / "source.json"
    source = destination if complete.exists() else getattr(CFG, f"DINO_{kind.upper()}_SOURCE")
    model = AutoModel.from_pretrained(str(source), torch_dtype=dtype, attn_implementation="sdpa")
    if not complete.exists():
        destination.mkdir(exist_ok=True)
        model.save_pretrained(destination, safe_serialization=True)
        atomic_json(complete, {"source": str(source), "revision": getattr(model.config, "_commit_hash", None)})
    return model

@torch.inference_mode()
def extract_features(frame):
    model = load_backbone("large", torch.float16).eval().to(DEVICE)
    model.requires_grad_(False)
    loader = make_loader(ViewDataset(frame, "crop"), CFG.FEATURE_BATCH)
    result = []
    try:
        for pixels in tqdm(loader, desc="DINOv2-Large crop + flip"):
            pixels = pixels.to(DEVICE, dtype=torch.float16, non_blocking=True)
            representations = []
            for image in [pixels, pixels.flip(-1)]:
                tokens = model(pixel_values=image).last_hidden_state.float()
                cls = F.normalize(tokens[:, 0], dim=1)
                patches = F.normalize(tokens[:, 1:].mean(1), dim=1)
                representations.append(torch.cat([cls, patches], dim=1) / math.sqrt(2))
            fused = F.normalize(torch.stack(representations).mean(0), dim=1)
            result.append(fused.cpu().numpy())
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()
    return np.concatenate(result).astype(np.float32)

combined = pd.concat([train, test], ignore_index=True)
sample_keys = np.array([f"train/{i}/{row.ID}" for i, row in train.iterrows()] +
                       [f"test/{row.ID}" for _, row in test.iterrows()])
feature_path = OUT / "large_crop_flip_features.npz"
if feature_path.exists():
    with np.load(feature_path, allow_pickle=False) as saved:
        assert str(saved["run_id"].item()) == RUN_ID
        assert np.array_equal(saved["keys"], sample_keys)
        features = saved["features"].copy()
    print("Loaded cached features:", features.shape)
else:
    features = extract_features(combined)
    temporary = feature_path.with_suffix(".tmp")
    with open(temporary, "wb") as handle:
        np.savez_compressed(handle, features=features, keys=sample_keys, run_id=RUN_ID)
    temporary.replace(feature_path)
assert features.shape == (len(combined), 2048) and np.isfinite(features).all()
assert np.allclose(np.linalg.norm(features, axis=1), 1.0, atol=1e-4)
X_train, X_test = features[:len(train)], features[len(train):]
y_train = train.label.to_numpy(dtype=int)


## 6. 五折 SVM 与 OOF 诊断

每折只使用其余四折训练 SVM，保留该折的 OOF 预测；测试概率对五个模型等权平均。
SVM 的 `C=10`、RBF 核和 `gamma='scale'` 沿用参考最终方案。
每个折模型单独保存，不在测试集上选择 SVM 参数。
随机分层折保留了重复拍摄场景的相关性，跨场景效果还需场景级分组验证。


In [ ]:
head_dir = OUT / "svm_folds"
head_dir.mkdir(exist_ok=True)
oof = np.zeros((len(train), len(CFG.CLASSES)), dtype=np.float64)
fold_test = []
fold_scores = []
for fold in tqdm(range(CFG.FOLDS), desc="Fitting SVM folds"):
    tr = np.flatnonzero(train.fold.to_numpy() != fold)
    va = np.flatnonzero(train.fold.to_numpy() == fold)
    destination = head_dir / f"fold_{fold}.joblib"
    if destination.exists():
        bundle = joblib.load(destination)
        assert bundle["run_id"] == RUN_ID and bundle["fold"] == fold
        estimator = bundle["model"]
    else:
        estimator = SVC(C=CFG.SVM_C, kernel="rbf", gamma="scale", probability=True,
                        random_state=CFG.FOLD_SEED, cache_size=512)
        estimator.fit(X_train[tr], y_train[tr])
        temporary = destination.with_suffix(".tmp")
        joblib.dump({"model": estimator, "run_id": RUN_ID, "fold": fold,
                     "classes": list(CFG.CLASSES)}, temporary)
        temporary.replace(destination)
    assert np.array_equal(estimator.classes_, np.arange(len(CFG.CLASSES)))
    oof[va] = estimator.predict_proba(X_train[va])
    fold_test.append(estimator.predict_proba(X_test))
    fold_scores.append({"fold": fold, "accuracy": accuracy_score(y_train[va], oof[va].argmax(1)),
                        "macro_f1": f1_score(y_train[va], oof[va].argmax(1), average="macro")})
base_probabilities = np.mean(fold_test, axis=0)
assert np.isfinite(base_probabilities).all() and np.allclose(base_probabilities.sum(1), 1)
base_labels = base_probabilities.argmax(1)
pd.DataFrame(fold_scores).to_csv(OUT / "svm_fold_scores.csv", index=False)
oof_metrics = {"accuracy": float(accuracy_score(y_train, oof.argmax(1))),
               "macro_f1": float(f1_score(y_train, oof.argmax(1), average="macro")),
               "scope": "frozen DINOv2-Large + five-fold SVM only; not full pipeline"}
atomic_json(OUT / "svm_oof_metrics.json", oof_metrics)
np.savez_compressed(OUT / "svm_probabilities.npz", oof=oof, test=base_probabilities,
                    train_keys=sample_keys[:len(train)], test_ids=test.ID.to_numpy(dtype=str))
print("SVM OOF only:", oof_metrics)
print(classification_report(y_train, oof.argmax(1), labels=range(len(CFG.CLASSES)),
                            target_names=CFG.CLASSES, digits=4, zero_division=0))
display(pd.DataFrame(fold_scores))
matrix = confusion_matrix(y_train, oof.argmax(1), labels=range(len(CFG.CLASSES)))
fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(matrix, cmap="Blues")
ax.set(xticks=range(5), yticks=range(5), xticklabels=CFG.CLASSES, yticklabels=CFG.CLASSES,
       xlabel="Predicted", ylabel="Actual", title="Frozen-feature SVM: OOF confusion matrix")
plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
for i in range(5):
    for j in range(5):
        ax.text(j, i, matrix[i, j], ha="center", va="center")
plt.tight_layout()
plt.show()


## 7. 草类专家 V2：残差头与分离的训练目标

只用原训练集中 Black-grass / Loose Silky-bent 的图片学习二分类。
每次随机选择 raw/crop/masked（0.3/0.4/0.3），使用轻微裁剪、水平/垂直翻转、90°旋转和颜色扰动。
前 3 轮只训练分类头；从第 4 轮开始，DINO 解冻最后 4 个 block，ConvNeXt 解冻卷积骨干。

两个专家分别保留第 15 / 10 轮 EMA 参数。训练到目标轮数后即停止，但余弦调度长度仍为原来的 27，
训练在既定日程内提前结束。这里记录**增强训练集准确率**。
使用混合精度、梯度累积、梯度裁剪与 EMA；每轮末保存可恢复的优化器、调度器和随机状态。

新分类头的两个分支分别承担以下目标：

- 主分支：`CE(baseline_logits, y)`。
- 新分支：`0.5 × [CE(detach(baseline_logits) + delta, y) + KL约束]`。
  KL 的教师为同一前向中的主分支，温度为 2；教师信号来自当前主分支。
- 新分支输入也使用 `detach()`，因此其交叉熵及 KL 梯度只更新新增参数。
  两组梯度分别裁剪，避免新增梯度通过统一范数缩放影响原分支。
- 新分支初始化使用独立的临时随机状态，不消耗主流程的随机数；分支内不用随机 Dropout。

原骨干仍由主分支训练、两分支都进入 EMA。训练准确率不用于选择测试输出。
初始化等价和梯度隔离只能降低改动范围；优化器数值、AMP 溢出等仍可能造成跨运行差异。
`baseline` 对照是本次模型关闭新分支后的输出。

自动求导语义：[PyTorch Tensor.detach](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.detach.html)。

接口参考：[ConvNeXt-Tiny](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.convnext_tiny.html)、
[PyTorch AMP](https://docs.pytorch.org/tutorials/recipes/recipes/amp_recipe.html)。


In [ ]:
class RandomQuarterTurn:
    def __call__(self, image):
        turns = [None, Image.Transpose.ROTATE_90, Image.Transpose.ROTATE_180, Image.Transpose.ROTATE_270]
        turn = random.choice(turns)
        return image if turn is None else image.transpose(turn)

class GrassDataset(Dataset):
    def __init__(self, frame, size):
        self.rows = frame.to_dict("records")
        self.transform = T.Compose([
            T.RandomResizedCrop(size, scale=(0.9, 1.0), ratio=(0.95, 1.05)),
            T.RandomHorizontalFlip(), T.RandomVerticalFlip(), RandomQuarterTurn(),
            T.RandomApply([T.ColorJitter(0.08, 0.08, 0.08)], p=0.4), T.ToTensor(), NORMALIZE,
        ])

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        view = random.choices(["raw", "crop", "masked"], weights=[0.3, 0.4, 0.3])[0]
        with Image.open(row[f"{view}_path"]) as image:
            pixels = self.transform(image.convert("RGB"))
        return pixels, int(row["grass_label"])

class ResidualLogitHead(nn.Module):
    """A linear baseline plus a zero-initialized, bounded nonlinear correction."""
    def __init__(self, in_features, dropout=0.0):
        super().__init__()
        # 先创建原主分支，保留相同种子下线性层的初始化顺序。
        self.baseline = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_features, 2))
        self.limit = float(CFG.LOGIT_CORRECTION_LIMIT)
        # 参数在 CPU 上创建；退出上下文时恢复 CPU RNG，避免影响主分支后续 Dropout。
        with torch.random.fork_rng(devices=[]):
            self.adapter = nn.Sequential(
                nn.LayerNorm(in_features), nn.Linear(in_features, CFG.ADAPTER_WIDTH),
                nn.GELU(), nn.Linear(CFG.ADAPTER_WIDTH, 2),
            )
            nn.init.zeros_(self.adapter[-1].weight)
            nn.init.zeros_(self.adapter[-1].bias)

    def forward(self, features, return_branches=False):
        baseline = self.baseline(features).float()
        delta = self.limit * torch.tanh(self.adapter(features.detach()).float())
        adapted = baseline + delta
        if return_branches:
            return {"baseline": baseline, "delta": delta, "adapted": adapted}
        return adapted

class DinoGrassClassifier(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        if pretrained:
            self.backbone = load_backbone("base", torch.float32)
        else:
            config = AutoConfig.from_pretrained(OUT / "dinov2_base")
            self.backbone = AutoModel.from_config(config, attn_implementation="sdpa")
        self.head = ResidualLogitHead(2 * self.backbone.config.hidden_size, dropout=0.15)

    def forward(self, pixels, return_branches=False):
        tokens = self.backbone(pixel_values=pixels).last_hidden_state
        features = torch.cat([tokens[:, 0], tokens[:, 1:].mean(1)], dim=1)
        return self.head(features, return_branches=return_branches)

class ConvNextGrassClassifier(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1 if pretrained and not CFG.CONVNEXT_WEIGHTS else None
        source = models.convnext_tiny(weights=weights)
        if pretrained and CFG.CONVNEXT_WEIGHTS:
            source.load_state_dict(torch.load(CFG.CONVNEXT_WEIGHTS, map_location="cpu", weights_only=True))
        self.features, self.avgpool = source.features, source.avgpool
        self.classifier = source.classifier
        width = self.classifier[2].in_features
        self.classifier[2] = ResidualLogitHead(width, dropout=0.0)

    def forward(self, pixels, return_branches=False):
        features = self.avgpool(self.features(pixels))
        features = self.classifier[1](self.classifier[0](features))
        return self.classifier[2](features, return_branches=return_branches)

def build_expert(arch, pretrained=True):
    if arch == "dino":
        return DinoGrassClassifier(pretrained=pretrained)
    if arch == "convnext":
        return ConvNextGrassClassifier(pretrained=pretrained)
    raise ValueError(f"Unknown expert: {arch}")

def expert_parameter_groups(model, arch):
    backbone = model.backbone if arch == "dino" else model.features
    head = model.head if arch == "dino" else model.classifier
    residual_head = model.head if arch == "dino" else model.classifier[2]
    adapter_parameters = list(residual_head.adapter.parameters())
    adapter_ids = {id(parameter) for parameter in adapter_parameters}
    head_parameters = [parameter for parameter in head.parameters() if id(parameter) not in adapter_ids]
    backbone_parameters = list(backbone.parameters())
    groups = [
        {"params": backbone_parameters, "lr": 1e-5 if arch == "dino" else 2e-5},
        {"params": head_parameters, "lr": 5e-4},
        {"params": adapter_parameters, "lr": CFG.ADAPTER_LR},
    ]
    core_parameters = backbone_parameters + head_parameters
    assert not ({id(p) for p in core_parameters} & adapter_ids)
    assert {id(p) for p in core_parameters + adapter_parameters} == {id(p) for p in model.parameters()}
    return groups, core_parameters, adapter_parameters

def expert_objective(branches, targets, criterion):
    baseline_loss = criterion(branches["baseline"], targets)
    # 不直接对 branches['adapted'] 做 CE，否则新增损失也会训练主分支。
    anchor = branches["baseline"].detach().float()
    adapted_for_loss = anchor + branches["delta"]
    residual_ce = criterion(adapted_for_loss, targets)
    temperature = CFG.DISTILL_TEMPERATURE
    kl = F.kl_div(F.log_softmax(adapted_for_loss / temperature, dim=1),
                  F.softmax(anchor / temperature, dim=1), reduction="batchmean") * temperature ** 2
    adaptation_loss = residual_ce + CFG.DISTILL_WEIGHT * kl
    total = baseline_loss + CFG.ADAPTER_LOSS_WEIGHT * adaptation_loss
    return total, baseline_loss, adaptation_loss

def set_trainable(model, arch, epoch):
    backbone = model.backbone if arch == "dino" else model.features
    backbone.requires_grad_(False)
    if epoch >= 4:
        if arch == "dino":
            for block in backbone.encoder.layer[-4:]:
                block.requires_grad_(True)
            backbone.layernorm.requires_grad_(True)
        else:
            backbone.requires_grad_(True)
    model.train()
    if epoch <= 3:
        backbone.eval()

def cpu_tree(value):
    if torch.is_tensor(value):
        return value.detach().cpu().clone()
    if isinstance(value, dict):
        return {key: cpu_tree(item) for key, item in value.items()}
    if isinstance(value, list):
        return [cpu_tree(item) for item in value]
    if isinstance(value, tuple):
        return tuple(cpu_tree(item) for item in value)
    return value

def rng_snapshot(generator):
    numpy_state = np.random.get_state()
    return {"python": random.getstate(), "numpy_kind": numpy_state[0],
            "numpy_keys": numpy_state[1].tolist(), "numpy_rest": numpy_state[2:],
            "torch": torch.get_rng_state(), "cuda": torch.cuda.get_rng_state_all(),
            "loader": generator.get_state()}

def restore_rng(state, generator):
    random.setstate(state["python"])
    np.random.set_state((state["numpy_kind"], np.array(state["numpy_keys"], dtype=np.uint32), *state["numpy_rest"]))
    torch.set_rng_state(state["torch"].cpu())
    for device_index, value in enumerate(state["cuda"][:torch.cuda.device_count()]):
        torch.cuda.set_rng_state(value.cpu(), device=device_index)
    generator.set_state(state["loader"].cpu())

@torch.no_grad()
def update_ema(average, model, step):
    decay = min(0.98, (1 + step) / (10 + step))
    for destination, source in zip(average.parameters(), model.parameters()):
        destination.lerp_(source, 1 - decay)
    for destination, source in zip(average.buffers(), model.buffers()):
        destination.copy_(source)

GRASS_INDICES = [CFG.CLASSES.index("Black-grass"), CFG.CLASSES.index("Loose Silky-bent")]
grass_train = train[train.label.isin(GRASS_INDICES)].copy().reset_index(drop=True)
grass_train["grass_label"] = grass_train.label.map({GRASS_INDICES[0]: 0, GRASS_INDICES[1]: 1})
assert set(grass_train.grass_label) == {0, 1}
print("Grass training samples:", len(grass_train))


In [ ]:
def train_expert(arch, epochs):
    directory = OUT / f"expert_{arch}"
    directory.mkdir(exist_ok=True)
    selected_path = directory / "selected.pt"
    latest_path = directory / "last.pt"
    if selected_path.exists():
        saved = torch.load(selected_path, map_location="cpu", weights_only=True)
        assert saved["run_id"] == RUN_ID and saved["epoch"] == epochs and saved["arch"] == arch
        del saved
        print(f"{arch}: completed checkpoint already exists.")
        return selected_path

    seed_everything(CFG.TRAIN_SEED)
    size = 392 if arch == "dino" else 384
    # 恢复时从本地配置构建模型，再加载完整断点，不再下载预训练参数。
    model = build_expert(arch, pretrained=not latest_path.exists()).to(DEVICE)
    set_trainable(model, arch, epoch=1)
    average = copy.deepcopy(model).eval().requires_grad_(False)
    groups, core_parameters, adapter_parameters = expert_parameter_groups(model, arch)
    optimizer = torch.optim.AdamW(groups, weight_decay=0.02)
    print(f"{arch}: added adapter parameters = {sum(p.numel() for p in adapter_parameters):,}")
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.COSINE_T_MAX, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda")
    criterion = nn.CrossEntropyLoss(label_smoothing=0.02)
    generator = torch.Generator().manual_seed(CFG.TRAIN_SEED)
    loader = make_loader(GrassDataset(grass_train, size), CFG.TRAIN_BATCH, shuffle=True, generator=generator)
    history, step, start_epoch = [], 0, 1
    if latest_path.exists():
        state = torch.load(latest_path, map_location="cpu", weights_only=True)
        assert state["run_id"] == RUN_ID and state["arch"] == arch
        model.load_state_dict(state["model"])
        average.load_state_dict(state["ema"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        scaler.load_state_dict(state["scaler"])
        history, step, start_epoch = state["history"], state["step"], state["epoch"] + 1
        restore_rng(state["rng"], generator)
        del state
        print(f"{arch}: continuing from epoch {start_epoch}")

    for epoch in range(start_epoch, epochs + 1):
        started = time.time()
        set_trainable(model, arch, epoch)
        optimizer.zero_grad(set_to_none=True)
        total_loss, total_correct, total = 0.0, 0, 0
        total_baseline_loss, total_adaptation_loss, baseline_correct, total_delta = 0.0, 0.0, 0, 0.0
        progress = tqdm(loader, desc=f"{arch} epoch {epoch}/{epochs}", leave=False)
        for batch_index, (pixels, targets) in enumerate(progress):
            pixels = pixels.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            # 按本次累积组实际样本数加权，正确处理末尾不足一个完整 batch 的情况。
            group_start = (batch_index // CFG.ACCUMULATION) * CFG.ACCUMULATION
            group_samples = min(CFG.TRAIN_BATCH * CFG.ACCUMULATION,
                                len(loader.dataset) - group_start * CFG.TRAIN_BATCH)
            with torch.amp.autocast("cuda", dtype=torch.float16):
                branches = model(pixels, return_branches=True)
                loss, baseline_loss, adaptation_loss = expert_objective(branches, targets, criterion)
            if not torch.isfinite(loss):
                raise RuntimeError(f"{arch} epoch {epoch}: non-finite loss")
            scaler.scale(loss * (len(targets) / group_samples)).backward()
            if (batch_index + 1) % CFG.ACCUMULATION == 0 or batch_index + 1 == len(loader):
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(core_parameters, 1.0)
                nn.utils.clip_grad_norm_(adapter_parameters, 1.0)
                previous_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                # 溢出时 GradScaler 会跳过优化，不应把该次计为 EMA 更新。
                if scaler.get_scale() >= previous_scale:
                    step += 1
                    update_ema(average, model, step)
            total_loss += loss.item() * len(targets)
            total_correct += (branches["adapted"].argmax(1) == targets).sum().item()
            baseline_correct += (branches["baseline"].argmax(1) == targets).sum().item()
            total_baseline_loss += baseline_loss.item() * len(targets)
            total_adaptation_loss += adaptation_loss.item() * len(targets)
            total_delta += branches["delta"].detach().abs().mean().item() * len(targets)
            total += len(targets)
            progress.set_postfix(loss=f"{total_loss / total:.4f}", train_acc=f"{total_correct / total:.3f}")
        if epoch > 3:
            scheduler.step()
        record = {"epoch": epoch, "loss": total_loss / total, "train_accuracy": total_correct / total,
                  "baseline_train_accuracy": baseline_correct / total,
                  "baseline_loss": total_baseline_loss / total, "adaptation_loss": total_adaptation_loss / total,
                  "mean_abs_logit_correction": total_delta / total,
                  "backbone_lr_next": optimizer.param_groups[0]["lr"], "seconds": round(time.time() - started, 1)}
        history.append(record)
        pd.DataFrame(history).to_csv(directory / "history.csv", index=False)
        atomic_torch_save(latest_path, {
            "run_id": RUN_ID, "arch": arch, "epoch": epoch, "step": step,
            "model": cpu_tree(model.state_dict()), "ema": cpu_tree(average.state_dict()),
            "optimizer": cpu_tree(optimizer.state_dict()), "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(), "history": history, "rng": rng_snapshot(generator),
        })
        print(arch, record, flush=True)

    atomic_torch_save(selected_path, {"run_id": RUN_ID, "arch": arch, "epoch": epochs,
                                      "head": "bounded_residual", "adapter_width": CFG.ADAPTER_WIDTH,
                                      "logit_correction_limit": CFG.LOGIT_CORRECTION_LIMIT,
                                      "size": size, "classes": [CFG.CLASSES[i] for i in GRASS_INDICES],
                                      "train_count": len(grass_train), "model": cpu_tree(average.state_dict())})
    del model, average, optimizer, scheduler, scaler, loader
    gc.collect()
    torch.cuda.empty_cache()
    return selected_path

expert_paths = {}
if CFG.TRAIN_EXPERTS:
    for arch, epochs in [("dino", CFG.DINO_EPOCHS), ("convnext", CFG.CONVNEXT_EPOCHS)]:
        expert_paths[arch] = train_expert(arch, epochs)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3))
    for arch in expert_paths:
        history = pd.read_csv(OUT / f"expert_{arch}" / "history.csv")
        axes[0].plot(history.epoch, history.loss, label=arch)
        axes[1].plot(history.epoch, history.train_accuracy, label=arch)
    axes[0].set_title("Augmented training loss")
    axes[1].set_title("Augmented training accuracy (not validation)")
    for axis in axes:
        axis.set_xlabel("Epoch")
        axis.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Grass expert training is disabled.")


## 8. 专家融合与成对预测

只有 SVM 判为两种草之一的样本才进入专家。DINO 使用 raw 视角、ConvNeXt 使用 masked 视角，
两者的 `P(Loose Silky-bent)` 等权平均，再应用统一阈值 0.25。
为保留参考配方，这里没有额外添加专家翻转 TTA。这个门控策略无法修正 SVM 已误分到非草类的样本。
每个专家在同一次前向中输出 baseline / adapted 两套概率。分别融合后应用相同的阈值，
得到严格按 ID 对齐的对照。两套结果使用同一骨干、同一 EMA 时刻、同一输入视角。


In [ ]:
@torch.inference_mode()
def predict_expert(checkpoint_path, frame, view):
    if len(frame) == 0:
        return {key: np.empty((0, 2), dtype=np.float32) for key in ["baseline", "adapted"]}
    state = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    assert state["run_id"] == RUN_ID
    assert state["classes"] == [CFG.CLASSES[i] for i in GRASS_INDICES]
    assert state["head"] == "bounded_residual" and state["adapter_width"] == CFG.ADAPTER_WIDTH
    assert state["logit_correction_limit"] == CFG.LOGIT_CORRECTION_LIMIT
    model = build_expert(state["arch"], pretrained=False)
    model.load_state_dict(state["model"])
    model.eval().to(DEVICE)
    loader = make_loader(ViewDataset(frame, view, state["size"]), CFG.TRAIN_BATCH)
    result = {key: [] for key in ["baseline", "adapted"]}
    for pixels in tqdm(loader, desc=f"Predicting {state['arch']} / {view}"):
        with torch.amp.autocast("cuda", dtype=torch.float16):
            branches = model(pixels.to(DEVICE, non_blocking=True), return_branches=True)
        for key in result:
            result[key].append(branches[key].float().softmax(1).cpu().numpy())
    del state, model, loader
    gc.collect()
    torch.cuda.empty_cache()
    return {key: np.concatenate(values) for key, values in result.items()}

def mix_paired_experts(dino, convnext, limit):
    for expert in [dino, convnext]:
        assert expert["baseline"].shape == expert["adapted"].shape
        for values in expert.values():
            assert values.ndim == 2 and values.shape[1] == 2
            assert np.isfinite(values).all() and ((values >= 0) & (values <= 1)).all()
            assert np.allclose(values.sum(1), 1.0, atol=1e-5)
    assert dino["baseline"].shape == convnext["baseline"].shape
    mixed = {key: 0.5 * dino[key][:, 1] + 0.5 * convnext[key][:, 1] for key in ["baseline", "adapted"]}
    # 每个二分类 logit 修正 ≤ limit，log-odds 变化 ≤ 2*limit。
    # sigmoid 在该最大位移下的全局最大差为 tanh(limit/2)。
    bound = math.tanh(limit / 2)
    if len(mixed["baseline"]):
        assert np.max(np.abs(mixed["adapted"] - mixed["baseline"])) <= bound + 1e-5
    return mixed, bound

def apply_grass_experts(base, positions, loose_probabilities, threshold, grass_indices):
    result = np.asarray(base, dtype=int).copy()
    positions = np.asarray(positions, dtype=int)
    probability = np.asarray(loose_probabilities)
    assert probability.shape == positions.shape and np.isfinite(probability).all()
    assert ((0 <= probability) & (probability <= 1)).all()
    assert np.isin(result[positions], grass_indices).all()
    result[positions] = np.where(probability >= threshold, grass_indices[1], grass_indices[0])
    return result

grass_positions = np.flatnonzero(np.isin(base_labels, GRASS_INDICES))
expert_labels = base_labels.copy()
baseline_expert_labels = base_labels.copy()
grass_probability = np.full(len(test), np.nan)
baseline_grass_probability = np.full(len(test), np.nan)
PROBABILITY_SHIFT_BOUND = math.tanh(CFG.LOGIT_CORRECTION_LIMIT / 2)
if CFG.TRAIN_EXPERTS and len(grass_positions):
    selected = test.iloc[grass_positions].reset_index(drop=True)
    expert_cache = OUT / "grass_paired_probabilities.npz"
    if expert_cache.exists():
        with np.load(expert_cache, allow_pickle=False) as saved:
            assert np.array_equal(saved["ids"], selected.ID.to_numpy(dtype=str))
            dino_probability = {key: saved[f"dino_{key}"].copy() for key in ["baseline", "adapted"]}
            convnext_probability = {key: saved[f"convnext_{key}"].copy() for key in ["baseline", "adapted"]}
    else:
        dino_probability = predict_expert(expert_paths["dino"], selected, "raw")
        convnext_probability = predict_expert(expert_paths["convnext"], selected, "masked")
        temporary = expert_cache.with_suffix(".tmp")
        with open(temporary, "wb") as handle:
            np.savez_compressed(handle, ids=selected.ID.to_numpy(dtype=str),
                                **{f"dino_{key}": values for key, values in dino_probability.items()},
                                **{f"convnext_{key}": values for key, values in convnext_probability.items()})
        temporary.replace(expert_cache)
    mixture, PROBABILITY_SHIFT_BOUND = mix_paired_experts(dino_probability, convnext_probability,
                                                        CFG.LOGIT_CORRECTION_LIMIT)
    grass_probability[grass_positions] = mixture["adapted"]
    baseline_grass_probability[grass_positions] = mixture["baseline"]
    expert_labels = apply_grass_experts(base_labels, grass_positions, mixture["adapted"], CFG.GRASS_THRESHOLD, GRASS_INDICES)
    baseline_expert_labels = apply_grass_experts(base_labels, grass_positions, mixture["baseline"], CFG.GRASS_THRESHOLD, GRASS_INDICES)
    outside_boundary = np.abs(mixture["baseline"] - CFG.GRASS_THRESHOLD) > PROBABILITY_SHIFT_BOUND + 1e-5
    assert np.array_equal(expert_labels[grass_positions[outside_boundary]],
                          baseline_expert_labels[grass_positions[outside_boundary]])
print(f"Grass gate: {len(grass_positions)}/{len(test)}; expert changes: {(expert_labels != base_labels).sum()}")
print("Head-only category changes before retrieval:", int((expert_labels != baseline_expert_labels).sum()))
print("Per-image probability-shift bound:", PROBABILITY_SHIFT_BOUND)


## 9. RootSIFT 与几何检索

从全部带标签训练图片提取局部描述子，以 FLANN 找到候选图，再通过 Lowe ratio test 与 RANSAC 单应性验证。
匹配分数为 `inliers² / matches`。只有第一名分数 ≥ 10、且 ≥ 第二名的 1.5 倍时才允许覆盖草类结果。
候选类别必须也是草类，且该测试样本必须原本通过 SVM 草类门控。

此方法会利用相同盆土、背景或重复拍摄场景，与纯植物形态识别不同。默认开启以沿用原方案；
若希望观察它的影响，对比输出中的 `submission_experts.csv` 和最终 `submission.csv`。
没有描述子、候选不足、退化的几何匹配都会自动跳过。
参考：[OpenCV 特征匹配与单应性](https://docs.opencv.org/4.x/d1/de0/tutorial_py_feature_homography.html)。


In [ ]:
def rootsift(path, sift):
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f"Cannot read image: {path}")
    scale = min(1.0, 1000 / max(image.shape))
    if scale < 1:
        image = cv2.resize(image, None, fx=scale, fy=scale)
    keypoints, descriptors = sift.detectAndCompute(image, None)
    if descriptors is None:
        return np.empty((0, 2), np.float32), np.empty((0, 128), np.float32)
    descriptors = np.sqrt(descriptors / (descriptors.sum(1, keepdims=True) + 1e-8))
    return np.float32([keypoint.pt for keypoint in keypoints]), np.ascontiguousarray(descriptors, dtype=np.float32)

def geometric_retrieval(train_frame, query_frame, num_classes):
    cv2.setRNGSeed(CFG.TRAIN_SEED)
    sift = cv2.SIFT_create(nfeatures=1100, contrastThreshold=0.02)
    reference = [rootsift(path, sift) for path in tqdm(train_frame.path, desc="RootSIFT training images")]
    valid_references = [i for i, (_, descriptors) in enumerate(reference) if len(descriptors) >= 2]
    results = []
    if valid_references:
        index = cv2.FlannBasedMatcher(dict(algorithm=1, trees=6), dict(checks=96))
        index.add([reference[i][1] for i in valid_references])
        index.train()
        exact = cv2.BFMatcher(cv2.NORM_L2)
    for row in tqdm(query_frame.itertuples(index=False), total=len(query_frame), desc="Matching test scenes"):
        best_by_class = [{"score": 0.0, "inliers": 0, "matches": 0, "train_id": None} for _ in range(num_classes)]
        query_points, query_descriptors = rootsift(row.path, sift)
        if valid_references and len(query_descriptors) >= 4:
            votes = np.zeros(len(train_frame), dtype=int)
            for pair in index.knnMatch(query_descriptors, k=2):
                if len(pair) == 2 and pair[0].distance < 0.85 * pair[1].distance:
                    votes[valid_references[pair[0].imgIdx]] += 1
            # 排序固定，避免 Python set 迭代次序影响 RANSAC 随机过程。
            candidates = sorted(valid_references, key=lambda i: (-votes[i], i))[:15]
            for candidate in candidates:
                points, descriptors = reference[candidate]
                good = [pair[0] for pair in exact.knnMatch(query_descriptors, descriptors, k=2)
                        if len(pair) == 2 and pair[0].distance < 0.75 * pair[1].distance]
                if len(good) < 4:
                    continue
                source = np.float32([query_points[match.queryIdx] for match in good])
                target = np.float32([points[match.trainIdx] for match in good])
                if len(np.unique(source, axis=0)) < 4 or len(np.unique(target, axis=0)) < 4:
                    continue
                matrix, mask = cv2.findHomography(source, target, cv2.RANSAC, 4.0, maxIters=3000, confidence=0.995)
                if matrix is None or mask is None or not np.isfinite(matrix).all():
                    continue
                inliers = int(mask.sum())
                if inliers < 4:
                    continue
                score = inliers * inliers / len(good)
                category = int(train_frame.iloc[candidate].label)
                if score > best_by_class[category]["score"]:
                    best_by_class[category] = {"score": float(score), "inliers": inliers,
                                               "matches": len(good), "train_id": str(train_frame.iloc[candidate].ID)}
        results.append({"ID": str(row.ID), "class_matches": best_by_class})
    return results

def apply_retrieval(current, base, ids, matches, grass_indices, min_score, margin):
    result = np.asarray(current, dtype=int).copy()
    positions = {name: i for i, name in enumerate(ids)}
    assert len(positions) == len(ids)
    assert len(matches) == len(ids) and {row["ID"] for row in matches} == set(ids)
    audit = []
    for row in matches:
        i = positions[row["ID"]]
        ranking = sorted(enumerate(row["class_matches"]), key=lambda item: (-item[1]["score"], item[0]))
        label, best = ranking[0]
        second_score = ranking[1][1]["score"]
        accepted = bool(base[i] in grass_indices and label in grass_indices and
                        best["score"] >= min_score and best["score"] >= margin * second_score)
        previous = int(result[i])
        if accepted:
            result[i] = label
        audit.append({"ID": row["ID"], "retrieval_label": int(label), "retrieval_score": best["score"],
                      "retrieval_second_score": second_score, "matched_train_id": best["train_id"],
                      "retrieval_accepted": accepted, "retrieval_changed": bool(accepted and label != previous)})
    return result, audit

final_labels = expert_labels.copy()
baseline_final_labels = baseline_expert_labels.copy()
retrieval_audit = []
if CFG.USE_RETRIEVAL and len(grass_positions):
    retrieval_path = OUT / "retrieval.json"
    if retrieval_path.exists():
        retrieval = json.loads(retrieval_path.read_text())
        assert retrieval["run_id"] == RUN_ID
        matches = retrieval["results"]
    else:
        matches = geometric_retrieval(train, test, len(CFG.CLASSES))
        atomic_json(retrieval_path, {"run_id": RUN_ID, "results": matches})
    final_labels, retrieval_audit = apply_retrieval(expert_labels, base_labels, test.ID.tolist(), matches,
                                                  GRASS_INDICES, CFG.MATCH_MIN_SCORE, CFG.MATCH_MARGIN)
    baseline_final_labels, _ = apply_retrieval(baseline_expert_labels, base_labels, test.ID.tolist(), matches,
                                             GRASS_INDICES, CFG.MATCH_MIN_SCORE, CFG.MATCH_MARGIN)
print("Retrieval changes:", int((final_labels != expert_labels).sum()))


## 10. 导出与检查

两套提交都只有 `ID,Category` 两列，按模板顺序输出。
`submission.csv` 与 `submission_v2.csv` 相同，启用残差修正；`submission_baseline.csv` 关闭修正。
`head_ablation_changes.csv` 列出两套最终输出的差异；零行表示本次残差分支保持了全部最终类别。

保存 Kaggle Version 时保留输出文件，可复用模型或通过第 2 节的 `RESUME_FROM` 恢复。


In [ ]:
def make_submission(ids, labels, classes):
    labels = np.asarray(labels, dtype=int)
    assert len(ids) == len(labels) and len(set(ids)) == len(ids)
    assert ((labels >= 0) & (labels < len(classes))).all()
    return pd.DataFrame({"ID": list(ids), "Category": [classes[i] for i in labels]})

for name, labels in [("features", base_labels), ("experts", expert_labels), ("final", final_labels),
                     ("baseline_experts", baseline_expert_labels), ("baseline", baseline_final_labels)]:
    make_submission(template_ids, labels, CFG.CLASSES).to_csv(OUT / f"submission_{name}.csv", index=False)
submission = make_submission(template_ids, final_labels, CFG.CLASSES)
submission_path = Path("/kaggle/working/submission.csv")
submission.to_csv(submission_path, index=False)
submission.to_csv(submission_path.with_name("submission_v2.csv"), index=False)
baseline_submission = make_submission(template_ids, baseline_final_labels, CFG.CLASSES)
baseline_submission.to_csv(submission_path.with_name("submission_baseline.csv"), index=False)
check = pd.read_csv(submission_path, dtype=str)
assert check.columns.tolist() == ["ID", "Category"]
assert check.ID.tolist() == template_ids and check.ID.is_unique
assert len(check) == len(test) and not check.isna().any().any()
assert set(check.Category).issubset(CFG.CLASSES)

audit = pd.DataFrame({"ID": template_ids,
                      "svm_category": [CFG.CLASSES[i] for i in base_labels],
                      "svm_confidence": base_probabilities.max(1),
                      "grass_gate": np.isin(base_labels, GRASS_INDICES),
                      "expert_p_loose": grass_probability,
                      "baseline_p_loose": baseline_grass_probability,
                      "probability_shift": grass_probability - baseline_grass_probability,
                      "expert_category": [CFG.CLASSES[i] for i in expert_labels],
                      "baseline_final_category": baseline_submission.Category,
                      "head_changed_before_retrieval": expert_labels != baseline_expert_labels,
                      "head_changed_final": final_labels != baseline_final_labels,
                      "final_category": submission.Category})
if retrieval_audit:
    audit = audit.merge(pd.DataFrame(retrieval_audit), on="ID", how="left", validate="one_to_one", sort=False)
    assert audit.ID.tolist() == template_ids
audit.to_csv(OUT / "prediction_audit.csv", index=False)
head_changes = audit[audit.head_changed_final].copy()
head_changes.to_csv(OUT / "head_ablation_changes.csv", index=False)
finite_shifts = audit.probability_shift.dropna().abs()
comparison = {"description": 'same EMA models, residual head enabled vs disabled; ',
              "added_head": "LayerNorm -> Linear(64 by default) -> GELU -> Linear(2) -> bounded tanh",
              "per_logit_limit": CFG.LOGIT_CORRECTION_LIMIT,
              "probability_shift_bound": PROBABILITY_SHIFT_BOUND,
              "max_observed_probability_shift": float(finite_shifts.max()) if len(finite_shifts) else 0.0,
              "changed_before_retrieval": int((expert_labels != baseline_expert_labels).sum()),
              "changed_after_retrieval": len(head_changes), "accuracy_difference": None}
atomic_json(OUT / "head_ablation_summary.json", comparison)
recipe = {"run_id": RUN_ID, "feature_model": "dinov2-large", "view": "crop_flip_average",
          "classifier": "five-fold RBF SVM", "svm_C": CFG.SVM_C,
          "experts_enabled": CFG.TRAIN_EXPERTS,
          "head": "zero-initialized bounded residual MLP + linear branch",
          "adapter_width": CFG.ADAPTER_WIDTH, "logit_correction_limit": CFG.LOGIT_CORRECTION_LIMIT,
          "adapter_lr": CFG.ADAPTER_LR, "adapter_loss_weight": CFG.ADAPTER_LOSS_WEIGHT,
          "distill_weight": CFG.DISTILL_WEIGHT, "distill_temperature": CFG.DISTILL_TEMPERATURE,
          "adapter_backbone_gradient": False,
          "experts": [{"arch": "dino", "epoch": CFG.DINO_EPOCHS, "view": "raw", "weight": 0.5},
                      {"arch": "convnext", "epoch": CFG.CONVNEXT_EPOCHS, "view": "masked", "weight": 0.5}],
          "grass_threshold": CFG.GRASS_THRESHOLD, "retrieval": CFG.USE_RETRIEVAL,
          "retrieval_min_score": CFG.MATCH_MIN_SCORE, "retrieval_margin": CFG.MATCH_MARGIN,
          "recipe_origin": "local residual-head modification of supplied V3; accuracy not validated",
          "full_pipeline_independent_accuracy": None, "test_count": len(test),
          "submission_sha256": hash_file(submission_path)}
atomic_json(OUT / "recipe.json", recipe)
display(submission.head(10))
display(submission.Category.value_counts().rename("test_predictions").to_frame())
print("Complete:", submission_path)
print("Models, checkpoints, OOF metrics and prediction audit:", OUT)
print("Residual head comparison:", comparison)
display(head_changes[["ID", "baseline_final_category", "final_category", "baseline_p_loose", "expert_p_loose"]])
display(FileLink(str(submission_path)))
display(FileLink(str(submission_path.with_name("submission_baseline.csv"))))
